In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
import torchaudio
import numpy as np
import os

In [2]:
class SpeechDataset(Dataset):
  def __init__(self, mix_dir, s1_dir, s2_dir, sample_rate=16000, num_samples=32000):
    self.mix_dir = mix_dir
    self.s1_dir = s1_dir
    self.s2_dir = s2_dir
    self.sample_rate = sample_rate
    self.num_samples = num_samples

    # Assuming all files have the same name for mix, s1, and s2
    self.ids = [os.path.splitext(f)[0] for f in os.listdir(mix_dir) if f.endswith('.wav')]

  def __len__(self):
    return len(self.ids)

  def _load_audio(self, path):
    wav, sr = torchaudio.load(path)
    wav = wav.mean(0)  # Convert to mono if necessary
    if sr != self.sample_rate:
      wav = torchaudio.functional.resample(wav, sr, self.sample_rate)
    if wav.shape[0] < self.num_samples:
      wav = F.pad(wav, (0, self.num_samples - wav.shape[0]))
    else:
      wav = wav[:self.num_samples]
    return wav

  def __getitem__(self, idx):
    utt_id = self.ids[idx]
    mix_wav = self._load_audio(os.path.join(self.mix_dir, utt_id + '.wav'))
    s1_wav  = self._load_audio(os.path.join(self.s1_dir, utt_id + '.wav'))
    s2_wav  = self._load_audio(os.path.join(self.s2_dir, utt_id + '.wav'))
    sources = torch.stack([s1_wav, s2_wav], dim=0)  # (2, T)
    return mix_wav, sources

In [16]:
mix_dir = "./Dataset/Libri2Mix/train-100/mix_clean"
s1_dir = "./Dataset/Libri2Mix/train-100/s1"
s2_dir = "./Dataset/Libri2Mix/train-100/s2"

dataset = SpeechDataset(
  mix_dir,
  s1_dir,
  s2_dir,
  sample_rate=8000,
  num_samples=30000
)

small_dataset = Subset(
  dataset,
  range(2000)
)

dataloader = DataLoader(
  small_dataset,
  batch_size=4,
  shuffle=True
)

In [17]:
dataset[0][0].shape, dataset[0][1].shape

(torch.Size([30000]), torch.Size([2, 30000]))

1. Define a simple DPRNN-based separator model

In [18]:
import torch.nn as nn
import torch

class DPRNNBlock(nn.Module):
  def __init__(self, input_size, hidden_size, chunk_size):
    super(DPRNNBlock, self).__init__()
    self.chunk_size = chunk_size
    self.intra_rnn = nn.LSTM(input_size, hidden_size, batch_first=True, bidirectional=True)
    self.intra_linear = nn.Linear(hidden_size * 2, input_size)
    self.intra_norm = nn.LayerNorm(input_size)

    self.inter_rnn = nn.LSTM(input_size, hidden_size, batch_first=True, bidirectional=True)
    self.inter_linear = nn.Linear(hidden_size * 2, input_size)
    self.inter_norm = nn.LayerNorm(input_size)

  def forward(self, x):
    # x: [batch, seq_len, feature_dim]
    B, T, F = x.size()
    # Chunking
    P = self.chunk_size
    num_chunks = (T + P - 1) // P  # ceil division
    pad_len = num_chunks * P - T
    if pad_len > 0:
      x = torch.cat([x, torch.zeros(B, pad_len, F, device=x.device)], dim=1)
    x = x.view(B, num_chunks, P, F)  # [B, num_chunks, chunk_size, F]

    # Intra-chunk RNN
    x_intra = x.contiguous().view(B * num_chunks, P, F)
    x_intra, _ = self.intra_rnn(x_intra)
    x_intra = self.intra_linear(x_intra)
    x_intra = self.intra_norm(x_intra)
    x_intra = x_intra + x.contiguous().view(B * num_chunks, P, F)  # Residual
    x_intra = x_intra.view(B, num_chunks, P, F)

    # Permute for inter-chunk RNN: [B, chunk_size, num_chunks, F]
    x_inter = x_intra.permute(0, 2, 1, 3).contiguous()
    x_inter = x_inter.view(B * P, num_chunks, F)
    x_inter, _ = self.inter_rnn(x_inter)
    x_inter = self.inter_linear(x_inter)
    x_inter = self.inter_norm(x_inter)
    x_inter = x_inter + x_intra.permute(0, 2, 1, 3).contiguous().view(B * P, num_chunks, F)  # Residual
    x_inter = x_inter.view(B, P, num_chunks, F).permute(0, 2, 1, 3).contiguous()  # [B, num_chunks, P, F]

    # Merge chunks
    x_out = x_inter.view(B, num_chunks * P, F)
    x_out = x_out[:, :T, :]  # remove padding if any
    return x_out

In [6]:
class DPRNNSeparator(nn.Module):
  def __init__(self, input_size, hidden_size, chunk_size, num_blocks):
    super(DPRNNSeparator, self).__init__()
    self.blocks = nn.ModuleList([
      DPRNNBlock(input_size, hidden_size, chunk_size) for _ in range(num_blocks)
    ])

  def forward(self, x):
    for block in self.blocks:
      x = block(x)
    return x

In [7]:
class DPRNN(nn.Module):
  def __init__(self, enc_dim=64, hidden_size=128, chunk_size=100, num_blocks=6, num_sources=2):
    super().__init__()
    self.encoder = nn.Conv1d(1, enc_dim, kernel_size=16, stride=8, padding=4)  # (B, 1, T) -> (B, E, L)
    self.separator = DPRNNSeparator(enc_dim, hidden_size, chunk_size, num_blocks)
    self.mask_net = nn.Sequential(
      nn.Conv1d(enc_dim, enc_dim * num_sources, 1),
      nn.ReLU()
    )
    self.decoder = nn.ConvTranspose1d(enc_dim, 1, kernel_size=16, stride=8, padding=4)
    self.num_sources = num_sources
    self.enc_dim = enc_dim

  def forward(self, mixture):
    # mixture: (B, T)
    x = mixture.unsqueeze(1)  # (B, 1, T)
    enc_out = self.encoder(x) # (B, E, L)
    # DPRNN expects (B, L, E)
    sep_in = enc_out.transpose(1, 2)  # (B, L, E)
    sep_out = self.separator(sep_in)  # (B, L, E)
    sep_out = sep_out.transpose(1, 2) # (B, E, L)
    # Predict masks for each source
    masks = self.mask_net(sep_out)    # (B, E * num_sources, L)
    B, _, L = masks.shape
    masks = masks.view(B, self.num_sources, self.enc_dim, L)
    masked = masks * enc_out.unsqueeze(1)  # (B, num_sources, E, L)
    # Decode
    est_sources = self.decoder(masked.view(B * self.num_sources, self.enc_dim, L)) # (B*num_sources, 1, T)
    est_sources = est_sources.squeeze(1).view(B, self.num_sources, -1) # (B, num_sources, T)
    return est_sources

In [8]:
def si_snr(est, ref, eps=1e-8):
  ref = ref - ref.mean(dim=1, keepdim=True)
  est = est - est.mean(dim=1, keepdim=True)
  s_target = (torch.sum(est * ref, dim=1, keepdim=True) * ref) / (
      torch.sum(ref ** 2, dim=1, keepdim=True) + eps)
  e_noise = est - s_target
  si_snr_val = 10 * torch.log10(
      (torch.sum(s_target ** 2, dim=1) + eps) /
      (torch.sum(e_noise ** 2, dim=1) + eps))
  return si_snr_val

def pit_loss(ests, refs):
  # ests/refs: (B, 2, T)
  loss1 = si_snr(ests[:, 0], refs[:, 0]) + si_snr(ests[:, 1], refs[:, 1])
  loss2 = si_snr(ests[:, 0], refs[:, 1]) + si_snr(ests[:, 1], refs[:, 0])
  # maximize SI-SNR, so minimize negative SI-SNR
  return -torch.mean(torch.stack([loss1, loss2], dim=0).max(dim=0)[0]) / 2

2. Instantiate model, optimizer, etc.

In [19]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DPRNN(
  enc_dim=64, hidden_size=128, chunk_size=100, num_blocks=2, num_sources=2
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
# criterion = nn.MSELoss()  # For waveform regression

3. Training loop

In [20]:
num_epochs = 5

for epoch in range(num_epochs):
  model.train()
  total_loss = 0

  print(f"Epoch: {epoch+1}/{num_epochs}")
  i = 0

  for mix_wav, sources in dataloader:
    mix_wav = mix_wav.to(device)         # (B, T)
    sources = sources.to(device)         # (B, 2, T)
    optimizer.zero_grad()
    est_sources = model(mix_wav)         # (B, 2, T)

    min_len = min(est_sources.shape[-1], sources.shape[-1])
    # print(mix_wav.shape)
    # print(sources.shape, est_sources.shape, min_len)
    # break
    # loss = criterion(est_sources[:, :, :min_len], sources[:, :, :min_len])
    loss = pit_loss(est_sources[:, :, :min_len], sources[:, :, :min_len])
    print(f"{i + 1}: {loss}")
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
    i += 1

  print(f"Epoch {epoch + 1}, Loss: {total_loss/len(dataloader):.4f}")

# Optionally, save the model
torch.save(model.state_dict(), 'dprnn_sep_model.pth')

Epoch: 1/5
1: 37.264278411865234
2: 17.28528594970703
3: 11.998062133789062
4: 7.991132736206055
5: 8.538199424743652
6: 4.799221038818359
7: 5.151650428771973
8: 4.145252704620361
9: 2.269115924835205
10: 3.7557647228240967
11: 1.74345862865448
12: 1.927675485610962
13: 2.2829675674438477
14: 1.9503461122512817
15: 2.177501678466797
16: 2.6871228218078613
17: 1.4712023735046387
18: 1.3514704704284668
19: 2.006093740463257
20: 1.7007800340652466
21: 1.0153567790985107
22: 1.6399192810058594
23: 0.10343071818351746
24: 0.61146080493927
25: 0.9157536625862122
26: 0.9849117398262024
27: 1.9960534572601318
28: 1.5237544775009155
29: 1.1823071241378784
30: 0.4170956313610077
31: 1.0119678974151611
32: 0.896613597869873
33: 0.18259496986865997
34: 1.2745378017425537
35: 2.206693410873413
36: 0.847805380821228
37: 0.8874039649963379
38: 0.5915124416351318
39: 0.4586206078529358
40: 0.8931092023849487
41: 1.0527169704437256
42: 1.9970965385437012
43: -0.060031116008758545
44: 1.201220035552978

In [21]:
import IPython.display as ipd

model.eval()
with torch.no_grad():
  mix, sources = dataset[1]
  est_sources = model(mix.unsqueeze(0).to(device)).cpu().squeeze(0)
  print("Mixture")
  ipd.display(ipd.Audio(mix.numpy(), rate=8000))
  print("Estimated Speaker 1")
  ipd.display(ipd.Audio(est_sources[0].numpy(), rate=8000))
  print("Estimated Speaker 2")
  ipd.display(ipd.Audio(est_sources[1].numpy(), rate=8000))

Mixture


Estimated Speaker 1


Estimated Speaker 2
